In [5]:
# Definir:
# Contribuição do colaborador por empresa
# Contrapartida da empresa
# Conta global (apenas para a Sopro Alimentos)

# Perguntas que eu preciso responder:
# 1 - Eu posso liberar os pagamentos dos boletos das empresas? (Vertex, Sopro alimentos e Aurora Clínicas)
# Para isso: revisar a base e aplicar as regras de negócio, descobrir quem saiu, conferir os valores de depósito e reportar inconsistencias


#  - Responder a Camila, gestora do RH da Vertex:
# !!! Confirmar quando os funcionarios da Vertex sairam, pela regra de negócio a Vertex precisa pagar até o momento

# Importar as bases de dados
# normalizar as bases de dados
# Descobrir quem ainda está na empresa e quem já saiu
# Conferir os valores de depósito e contrapartida da empresa
# Gerar novo 

import pandas as pd

colaboradores = pd.read_csv("colaboradores.csv")
colaboradores = colaboradores.dropna()

# Carrega a base de colaboradores
df = pd.read_csv('colaboradores.csv')

# Agrupa por empresa
grupos = df.groupby('empresa')

# Separa colaboradores por empresa
df_vertex = grupos.get_group('Vertex Logística')
df_sopro = grupos.get_group('Sopro Alimentos')
df_aurora = grupos.get_group('Aurora Clínicas')

# display(df_vertex)
# display(df_sopro)
# display(df_aurora)


In [10]:
# descobrir quem ainda está na empresa e quem já saiu

# Data de referência: agosto de 2026
mes_referencia = pd.Period('2026-08', freq='M')

def analisar_desligados(df, nome_empresa):
    df = df.copy()
    df['data_desligamento'] = pd.to_datetime(df['data_desligamento'], errors='coerce')
    
    # Quem já saiu (independente do mês)
    desligados = df[df['status'] == 'desligado']
    print(f"\n=== {nome_empresa} — Desligados ===")
    print(desligados[['nome', 'data_desligamento']].to_string(index=False))
    
    # Regra: mantém na base quem saiu no mês de referência (ago/2026)
    saiu_no_mes_referencia = desligados['data_desligamento'].dt.to_period('M') == mes_referencia
    
    # Base final: todos os admitidos + desligados do mês de referência
    df_base_atualizada = df[(df['status'] == 'admitido') | 
                             ((df['status'] == 'desligado') & saiu_no_mes_referencia)]
    
    # Quem sai definitivamente da base (desligados antes do mês de referência)
    df_removidos = desligados[~saiu_no_mes_referencia]
    
    return df_base_atualizada, df_removidos

df_vertex_base, df_vertex_removidos = analisar_desligados(df_vertex, 'Vertex Logística')
df_sopro_base, df_sopro_removidos     = analisar_desligados(df_sopro, 'Sopro Alimentos')
df_aurora_base, df_aurora_removidos   = analisar_desligados(df_aurora, 'Aurora Clínicas')


=== Vertex Logística — Desligados ===
                   nome data_desligamento
     Ana Klein Oliveira        2026-07-18
Thiago Queiroz Oliveira        2026-07-29

=== Sopro Alimentos — Desligados ===
Empty DataFrame
Columns: [nome, data_desligamento]
Index: []

=== Aurora Clínicas — Desligados ===
                    nome data_desligamento
Beatriz Nogueira Ribeiro        2026-08-21
   Simone Fonseca Duarte        2026-08-27


In [ ]:
# exporta o resultado para um arquivo CSV
df_vertex_base.to_csv("colaboradores_vertex.csv", index=False)
df_sopro_base.to_csv("colaboradores_sopro.csv", index=False)
df_aurora_base.to_csv("colaboradores_aurora.csv", index=False)